In [6]:
!pip install jinja2

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp311-cp311-win_amd64.whl (15 kB)



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import sys
import os

# Trỏ đường dẫn về thư mục gốc để gọi các file trong thư mục src/
sys.path.append(os.path.abspath('../')) 

from src.data_loader import load_data
from src.preprocess import clean_text
from src.features import run_feature_pipeline
from src.train import train_mnb_model

import warnings
warnings.filterwarnings('ignore')

In [2]:
# 1. Đọc dữ liệu chuẩn
df = load_data('../data/raw/labeled_comments.xlsx')

# 2. Tiền xử lý
df['clean_text'] = df['Text'].apply(clean_text)

# 3. Vector hóa
feature_dict = run_feature_pipeline(df)
vectorizer = feature_dict["vectorizer"]
X_train_vec = feature_dict["X_train_tfidf"]
y_train = feature_dict["y_train"]

# 4. Huấn luyện nhanh
model = train_mnb_model(X_train_vec, y_train)

print("✅ Hệ thống đã sẵn sàng để nhận dữ liệu test!")

⏳ Đang tải dữ liệu từ: ../data/raw/labeled_comments.xlsx...
✅ Tải dữ liệu hoàn tất! Thu thập được 1242.
⏳ Đang huấn luyện mô hình Multinomial Naive Bayes...
✅ Huấn luyện hoàn tất!
✅ Hệ thống đã sẵn sàng để nhận dữ liệu test!


In [4]:
def test_model(comment):
    # Bước 1: Làm sạch câu văn vừa nhập
    cleaned_comment = clean_text(comment)
    
    # Bước 2: Dùng vectorizer đã học để biến câu văn thành ma trận số
    comment_vec = vectorizer.transform([cleaned_comment])
    
    # Bước 3: Đưa ma trận vào mô hình để dự đoán
    prediction = model.predict(comment_vec)[0]
    
    # Hiển thị kết quả đẹp mắt
    color = "🟢" if prediction == "positive" else "🔴"
    print(f"Bình luận: '{comment}'")
    print(f"AI Dự đoán: {color} [{prediction.upper()}]\n")
    print("-" * 50)

# ================= TEST THỬ VỚI CÁC TRƯỜNG HỢP KHÓ =================

# 1. Khen rõ ràng
test_model("Sản phẩm xài quá mượt, shipper thân thiện, cho shop 5 sao luôn!")

# 2. Chê rõ ràng
test_model("Máy mua về sạc không vào pin, nhắn tin shop không thèm trả lời, tẩy chay.")

# 3. Bình luận lắt léo (Có từ chê nhưng nghĩa là khen)
test_model("Xài con điện thoại này mượt mà tới mức không có gì để chê.")

# 4. Khen chê lẫn lộn
test_model("Ngoại hình máy khá đẹp nhưng pin tụt nhanh như tụt quần, thất vọng.")

Bình luận: 'Sản phẩm xài quá mượt, shipper thân thiện, cho shop 5 sao luôn!'
AI Dự đoán: 🟢 [POSITIVE]

--------------------------------------------------
Bình luận: 'Máy mua về sạc không vào pin, nhắn tin shop không thèm trả lời, tẩy chay.'
AI Dự đoán: 🔴 [NEGATIVE]

--------------------------------------------------
Bình luận: 'Xài con điện thoại này mượt mà tới mức không có gì để chê.'
AI Dự đoán: 🟢 [POSITIVE]

--------------------------------------------------
Bình luận: 'Ngoại hình máy khá đẹp nhưng pin tụt nhanh như tụt quần, thất vọng.'
AI Dự đoán: 🟢 [POSITIVE]

--------------------------------------------------


In [19]:
# Đọc file dữ liệu thô (Unseen Data)
df_raw = pd.read_csv('../data/raw/comments_data.csv')

# Lấy ngẫu nhiên 100 dòng để test
sample_df = df_raw.sample(100, random_state=42).copy()

# Chạy tiền xử lý và dự đoán
sample_df['clean_text'] = sample_df['Text'].apply(clean_text)
sample_vec = vectorizer.transform(sample_df['clean_text'])

# Lưu thẳng kết quả dự đoán vào cột tên là 'label' cho chuẩn
sample_df['label'] = model.predict(sample_vec)

# Tạo màu sắc cho bảng hiển thị
def color_prediction(val):
    color = '#d4edda' if val == 'positive' else '#f8d7da'
    return f'background-color: {color}; color: black'

# Hiển thị bảng kết quả
display_df = sample_df[['Text', 'label']]
display_df.style.map(color_prediction, subset=['label']).hide(axis='index')

Text,label
củ nghệ,positive
"Mình thích lắm nha, dây màu đẹp, chuẩn lắm, không có quá cứng, giá hợp lý",positive
tai nghe gia công tốt giá hợp lý còn độ bền phải để dùng 1 thời gian mới đánh giá được cho shop 5*,positive
"Sản phẩm gia công đẹp. Thiết kế mở nắp không tiện dụng lắm, phải giữ với 2 tay. Hy vọng bền bỉ. Giao hàng nhanh. Chúc năm mới Tiki trading làm ăn phát đạt!",positive
Mở ra thì sản phẩm đã bị bóc rồi. 1 bên hơi chập trờn. Dù sao cũng cho shop 4* mong lần sau shop cố gắng,positive
Always never strike lucky draw,negative
"Mua được 3tháng, chơi game không mượt bằng Xiaomi cùng tầm giá. Độ sáng màn hình hơi yếu. Còn lại ok tất. Đang mua nhé.",positive
chất_liệu chất_liệu ok,positive
"Đã nhận hàng, sản phẩm đóng gói còn nguyên hộp.",positive
đúng với mô_tả chuẩn,positive
